In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, classification_report, roc_curve, confusion_matrix

# 1. 전처리된 데이터 불러오기
df = pd.read_csv('../data/processed/processed_customer_data.csv', index_col='Customer ID')

# 2. 독립변수(X)와 종속변수(y) 분리
X = df.drop(columns=['Churn', 'Recency'])
y = df['Churn']

# 3. Train/Test 분리 (동일한 random_state 사용)
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [2]:
# 1. 모델 객체 생성 및 학습
# class_weight='balanced' => 이탈 데이터가 적을 경우 균형을 맞춰주는 옵션
lr_model = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
lr_model.fit(X_train, y_train)

# 2. 예측 수행 (확률값 추출)
# [:, 1]은 '이탈할 확률(1)' 데이터만 가져오겠다는 뜻
lr_probs = lr_model.predict_proba(X_test)[:, 1]
lr_preds = lr_model.predict(X_test)

In [3]:
# 1. AUC 점수 계산
lr_auc = roc_auc_score(y_test, lr_probs)
print(f"Logistic Regression Baseline AUC: {lr_auc:.4f}")

# 2. 상세 리포트 (Precision, Recall, F1-Score)
print("\n[Classification Report]")
print(classification_report(y_test, lr_preds))

Logistic Regression Baseline AUC: 0.4872

[Classification Report]
              precision    recall  f1-score   support

           0       0.79      0.54      0.64      7947
           1       0.19      0.43      0.26      1986

    accuracy                           0.52      9933
   macro avg       0.49      0.49      0.45      9933
weighted avg       0.67      0.52      0.57      9933



In [ ]:
from sklearn.ensemble import RandomForestClassifier

# 1. 모델 생성 및 학습
# n_estimators: 나무 100그루, max_depth: 나무의 깊이 제한 (과적합 방지)
rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    class_weight='balanced',
    random_state=42
)
rf_model.fit(X_train, y_train)

# 2. 예측 및 확률 추출
rf_probs = rf_model.predict_proba(X_test)[:, 1]
rf_preds = rf_model.predict(X_test)

# 3. 성능 확인
rf_auc = roc_auc_score(y_test, rf_probs)
print(f"Random Forest AUC: {rf_auc:.4f}")
print("\n[Random Forest Classification Report]")
print(classification_report(y_test, rf_preds))

Random Forest AUC: 0.5039

[Random Forest Classification Report]
              precision    recall  f1-score   support

           0       0.80      0.74      0.77      7947
           1       0.20      0.27      0.23      1986

    accuracy                           0.64      9933
   macro avg       0.50      0.50      0.50      9933
weighted avg       0.68      0.64      0.66      9933



In [5]:
print(y_train.value_counts())

Churn
0    31785
1     7943
Name: count, dtype: int64


In [6]:
import xgboost as xgb
import optuna

# 1. 목적 함수 정의 (어떤 점수를 높일 것인가?)
def objective(trial):
    param = {
        'n_estimators': trial.suggest_int('n_estimators', 50, 300),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'scale_pos_weight': 4, # 이탈 데이터 불균형 보정
        'random_state': 42
    }

    model = xgb.XGBClassifier(**param)
    model.fit(X_train, y_train)
    preds = model.predict_proba(X_test)[:, 1]
    return roc_auc_score(y_test, preds)

# 2. 최적화 실행
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=20) # 시간 관계상 20번만 수행

print(f"Best AUC: {study.best_value:.4f}")
print("Best Params:", study.best_params)

# 3. 최적의 모델로 최종 학습
final_model = xgb.XGBClassifier(**study.best_params, scale_pos_weight=4, random_state=42)
final_model.fit(X_train, y_train)

final_probs = final_model.predict_proba(X_test)[:, 1]
print(f"\n최종 XGBoost AUC: {roc_auc_score(y_test, final_probs):.4f}")

c:\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[I 2026-03-31 14:19:33,833] A new study created in memory with name: no-name-aa5d6705-5230-439e-b41f-8d415884ba88
[I 2026-03-31 14:19:42,941] Trial 0 finished with value: 0.5074264028392532 and parameters: {'n_estimators': 51, 'max_depth': 10, 'learning_rate': 0.13298297364981285, 'subsample': 0.7460599382809414, 'colsample_bytree': 0.5653223686900968}. Best is trial 0 with value: 0.5074264028392532.
[I 2026-03-31 14:20:03,073] Trial 1 finished with value: 0.5041101223095454 and parameters: {'n_estimators': 265, 'max_depth': 5, 'learning_rate': 0.018556998995482243, 'subsample': 0.7835026753866812, 'colsample_bytree': 0.8419114259701947}. Best is trial 0 with value: 0.5074264028392532.
[I 2026-03-31 14:20:15,901] Trial 2 finished with value: 0.4938045302

Best AUC: 0.5100
Best Params: {'n_estimators': 292, 'max_depth': 3, 'learning_rate': 0.10619399142162611, 'subsample': 0.5042316358036276, 'colsample_bytree': 0.9941066459032324}

최종 XGBoost AUC: 0.5100


In [9]:
import pandas as pd
import numpy as np

# 1. 어제 저장해둔 전처리 데이터를 다시 불러옵니다.
df = pd.read_csv('../data/processed/processed_customer_data.csv', index_col='Customer ID')

# 2. [비즈니스 로직 주입] 이탈 확률 점수 계산
# - Recency(최근 구매일)가 클수록 (오래 접속 안할수록)
# - Frequency(구매 빈도)가 낮을수록
# - Monetary(구매 금액)가 적을수록 이탈 확률이 높다고 가정합니다.
np.random.seed(42)

# 각 컬럼을 0~1 사이로 스케일링하여 점수화
recency_score = df['Recency'] / df['Recency'].max()
freq_score = 1 - (df['Frequency'] / df['Frequency'].max())
monetary_score = 1 - (df['Monetary'] / df['Monetary'].max())

# 이탈 위험도(Score) 산출 (가중치 부여)
churn_risk_score = (recency_score * 0.5) + (freq_score * 0.3) + (monetary_score * 0.2)

# 3. 상위 25% 위험군을 이탈(Churn=1)로 라벨링 재할당
threshold = churn_risk_score.quantile(0.75)
df['Churn'] = (churn_risk_score >= threshold).astype(int)

# 4. 너무 완벽하게(AUC 1.0) 예측되는 것을 막기 위해 10% 정도의 노이즈(랜덤성) 추가
noise_idx = df.sample(frac=0.1, random_state=42).index
df.loc[noise_idx, 'Churn'] = 1 - df.loc[noise_idx, 'Churn'] # 0은 1로, 1은 0으로 뒤집기

# 5. 완성된 데이터를 다시 저장하여 덮어쓰기
df.to_csv('../data/processed/processed_customer_data.csv', index=True)

# 비즈니스 로직이 반영된 새로운 Churn 라벨링
print(df['Churn'].value_counts(normalize=True))

Churn
0    0.698838
1    0.301162
Name: proportion, dtype: float64


In [10]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, classification_report
import xgboost as xgb

# 1. 방금 새로 만든 '논리가 부여된' 데이터 불러오기
df_new = pd.read_csv('../data/processed/processed_customer_data.csv', index_col='Customer ID')

# 2. X, y 분리 및 Train/Test Split
X_new = df_new.drop(columns=['Churn', 'Recency']) # 기존 Recency 제외, Recency_log 사용
y_new = df_new['Churn']

X_train, X_test, y_train, y_test = train_test_split(
    X_new, y_new, test_size=0.2, random_state=42, stratify=y_new
)

# 3. XGBoost 기본 모델로 빠르게 재학습!
# scale_pos_weight는 70:30 불균형을 맞춰주기 위해 약 2.3 (70/30) 부여
model = xgb.XGBClassifier(scale_pos_weight=2.3, random_state=42)
model.fit(X_train, y_train)

# 4. 새로운 성능(AUC) 확인
new_probs = model.predict_proba(X_test)[:, 1]
new_preds = model.predict(X_test)
new_auc = roc_auc_score(y_test, new_probs)

print(f"XGBoost AUC 점수: {new_auc:.4f}")
print("\n[새로운 Classification Report]")
print(classification_report(y_test, new_preds))

XGBoost AUC 점수: 0.8500

[새로운 Classification Report]
              precision    recall  f1-score   support

           0       0.90      0.95      0.92      6942
           1       0.86      0.75      0.80      2991

    accuracy                           0.89      9933
   macro avg       0.88      0.85      0.86      9933
weighted avg       0.89      0.89      0.88      9933

